# Imports

In [ ]:
from cytools import Polytope

In [ ]:
import sys; sys.path.append('..')
from src import cydata, Zp, diagnostics

In [ ]:
import sys; sys.path.append('../../cornell-dev')
from projects.kklt.kklt_lib import kklt_conifolds

In [ ]:
import numpy as np

# Load Manwe

In [ ]:
verifying_from_dsV1_repo = False

## From the dSv1 data file (for verification)

In [ ]:
if verifying_from_dsV1_repo:
    import gzip, pickle

    # load the dataframe
    with gzip.open('dSv1.p', 'rb') as f:
        dSv1_df = pickle.load(f)

    # read the CY data
    manwe_df = dSv1_df[dSv1_df['name']=='manwe'].iloc[0]
    p  = Polytope(manwe_df['dual points'])
    t  = p.triangulate(heights=manwe_df['mirror heights'])
    cy = t.cy()

## Hard-coded Manwe's CY

In [ ]:
if not verifying_from_dsV1_repo:
    verts   = [[0, 0, 0, 0], [1, -1, -1, -1], [-1, 2, 1, 1], [-1, -1, 0, 0], [-1, -1, 2, 0], [-1, -1, 2, 1], [-1, 0, 0, 2], [-1, -1, 0, 2], [-1, 0, 0, 1], [-1, 0, 1, 0], [-1, -1, 0, 1], [-1, -1, 1, 0], [-1, -1, 1, 1], [-1, 0, 1, 1], [-1, 1, 1, 1], [0, -1, 0, 0]]
    heights = [0, 35, 29, 35, 31, 35, 35, 35, 15, 17, 31, 9, 21]
    p       = Polytope(verts)
    t       = p.triangulate(heights=heights)
    cy      = t.cy()

## Set the conifold-related info

In [ ]:
# get the conifold charge
# -----------------------
conis = list(kklt_conifolds.kklt_conifolds(p.dual(), as_class=True))
assert len(conis) == 1
q = conis[0].conifold_charge()

# set the COB (same one used in paper)
cob = np.array([[0,0,0,0,0,-1,0,0],
[-1,0,0,0,0,0,0,0],
[0,-1,0,0,0,0,0,0],
[0,0,-1,0,0,0,0,0],
[0,0,0,-1,0,0,0,0],
[0,0,0,0,-1,0,0,0],
[0,0,0,0,0,0,-1,0],
[0,0,0,0,0,0,0,-1]])

## Hard-code Manwe

In [ ]:
data = cydata.CYData.from_cy(cy, coni_curve=q, coni_cob=cob)

K = np.array([-6, -1,   0, 1, -3,  2,  0, -1])
M = np.array([16, 10, -26, 8, 32, 30, 18, 28])
manwe = diagnostics.PFV(data, K=K, M=M)
print(manwe.check_all())

Set GVs (for fancier diagnostics)

In [ ]:
manwe.gvs = manwe.cy.compute_gvs(max_deg=10)
manwe.diagnostics()

# Find Manwe from scratch

Get some p-vectors

In [ ]:
ps = Zp.pvecs(data, max_deg = 1260)

Do the search!

In [ ]:
pfvs = Zp.coniZpM(
    data=data,
    ps=ps,
    Qmax=data.h11+data.h21+4,
    M0min=13,
    ellipsoid_dilation=200,
    use_box=False,
    max_N_pfvs=100_000_000,
    return_formal_pfvs=True,
    verbosity=0
)

In [ ]:
for i,pfv in enumerate(pfvs):
    if all(pfv.K == manwe.K) and all(pfv.M == manwe.M):
        print(f"we found manwe at index {i} :)")
        break

## Plot the search results

Assign GVs to all of the PFVs

In [ ]:
gvs = pfvs[0].cy.compute_gvs(max_deg=10).coo

for pfv in pfvs:
    pfv.gvs = gvs

Extract data of interest

In [ ]:
W0s    = [pfv.W0() for pfv in pfvs]
aligns = [pfv.align for pfv in pfvs]
gsMs   = [pfv.gsM for pfv in pfvs]

Plot it!

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(W0s, aligns, c=gsMs, s=4)
plt.scatter([manwe.W0()], [manwe.align], c=[manwe.gsM], marker='*', s=100)

plt.colorbar(label='gsM')

# axis scaling
plt.xscale('log')
plt.yscale('log')

# axis labels
plt.title("PFVs from Manwe's conifold")
plt.xlabel('W0')
plt.ylabel('align')
